<a href="https://colab.research.google.com/github/srinayani123/Arabic_TTS/blob/main/mms_tts_ara_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForTextToWaveform

tokenizer = AutoTokenizer.from_pretrained("facebook/mms-tts-ara")
model = AutoModelForTextToWaveform.from_pretrained("facebook/mms-tts-ara")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/288 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/460 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.64k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/145M [00:00<?, ?B/s]

In [ ]:
from transformers import VitsModel, AutoTokenizer
import torch
import time
import torchaudio

# Load Facebook MMS Arabic TTS
model = VitsModel.from_pretrained("facebook/mms-tts-ara")
tokenizer = AutoTokenizer.from_pretrained("facebook/mms-tts-ara")

# Sentences in MSA and Saudi dialects
texts = [
    {"label": "MSA", "text": "مرحبًا، هذه السيارة مزودة بمحرك توربو سعة 2.0 لتر ونظام ملاحة متقدم."},
    {"label": "Najdi", "text": "مرحبا، هالسيارة فيها محرك تيربو 2.0 لتر ونظام نافيجيشن متطور."},
    {"label": "Hijazi", "text": "هال، السيارة دي فيها محرك تيربو 2.0 لتر ونظام نافيجيشن زين."},
    {"label": "Gulf", "text": "هال، السيارة هذي فيها محرك تيربو 2.0 لتر ونظام نافيجيشن ممتاز."}
]

results = []

# Run TTS for each sentence
for idx, item in enumerate(texts):
    label = item["label"]
    text = item["text"]
    print(f"\n🔊 [{label}] Text {idx+1}: {text}")

    # Tokenize
    inputs = tokenizer(text, return_tensors="pt")

    # Latency timer
    start_time = time.time()
    with torch.no_grad():
        output = model(**inputs)
    end_time = time.time()

    # Save waveform
    waveform = output.waveform
    sample_rate = model.config.sampling_rate
    audio_path = f"tts_{label.lower()}.wav"
    torchaudio.save(audio_path, waveform, sample_rate)

    # Performance metrics
    duration_sec = waveform.shape[1] / sample_rate
    latency_sec = end_time - start_time

    results.append({
        "label": label,
        "text": text,
        "latency_sec": latency_sec,
        "duration_sec": duration_sec,
        "audio_path": audio_path
    })

    print(f"✅ Saved to: {audio_path}")
    print(f"⏱️ Latency: {latency_sec:.2f} sec | 🕒 Duration: {duration_sec:.2f} sec")



🔊 [MSA] Text 1: مرحبًا، هذه السيارة مزودة بمحرك توربو سعة 2.0 لتر ونظام ملاحة متقدم.
✅ Saved to: tts_msa.wav
⏱️ Latency: 10.85 sec | 🕒 Duration: 9.49 sec

🔊 [Najdi] Text 2: مرحبا، هالسيارة فيها محرك تيربو 2.0 لتر ونظام نافيجيشن متطور.
✅ Saved to: tts_najdi.wav
⏱️ Latency: 8.61 sec | 🕒 Duration: 7.79 sec

🔊 [Hijazi] Text 3: هال، السيارة دي فيها محرك تيربو 2.0 لتر ونظام نافيجيشن زين.
✅ Saved to: tts_hijazi.wav
⏱️ Latency: 9.52 sec | 🕒 Duration: 7.81 sec

🔊 [Gulf] Text 4: هال، السيارة هذي فيها محرك تيربو 2.0 لتر ونظام نافيجيشن ممتاز.
✅ Saved to: tts_gulf.wav
⏱️ Latency: 7.78 sec | 🕒 Duration: 7.23 sec
